# 10 · Supervised training
**Status: UNEXECUTED FIRST DRAFT.** No training result is claimed.

**Purpose:** train the complete AHR v2 XL model from observed transitions, with policy supervision only for explicit expert/human demonstrations. Queen, Worker, Scout, world model and trainable geometry remain present.

**Inputs:** notebooks 00–09, `config/ahr_v2_xl.json`, an explicit train/dev/heldout game-family split, validated JSONL transitions and demonstrations, passing runtime and full-model continuation gates. **Outputs:** checkpoints, JSONL learning curves and a run summary. **Expected runtime:** up to `train.max_wall_seconds` per invocation (default one hour), plus loading; step speed and fit are unmeasured. The model is not silently reduced if it does not fit.

**Checkpoint directory:** `checkpoints/<run_id>/`. **Exact resume:** keep the same run ID, configuration, notebook code and input files, then Run All again. The latest verified checkpoint restores optimizer, scheduler, scaler, RNG and sampler position. Increase the training target through a new explicit warm-start run if configuration changes.

**Run All behavior:** validates prerequisites and starts the real bounded training stage when they exist. Missing data/gates stop the stage with an actionable message. This file has only been authored and statically checked. One selected GPU is used per run; an eight-GPU campaign is configured in notebook 12.

## Load definitions

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([9], globals())


## Checkpoint and data identity
Warm starts load weights from a named, trusted project checkpoint. Resume is stricter: the checkpoint manager requires identical configuration and notebook source. Input hashes bind data order and labels to the optimizer trajectory.

In [ ]:
import copy
import contextlib
import math
import time
import torch
import torch.nn.functional as F


def read_ahr_checkpoint(checkpoint_path, root=ROOT, expected_sha256=None):
    path = Path(checkpoint_path).expanduser()
    if not path.is_absolute():
        path = Path(root) / path
    path = path.resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Checkpoint missing: {path}. Train a model first; no random model will be substituted.')
    digest = sha256_file(path)
    commit_path = path.with_name(path.stem + '.commit.json')
    if commit_path.is_file():
        commit = json.loads(commit_path.read_text())
        if commit.get('sha256') != digest or commit.get('file') != path.name:
            raise ValueError('Checkpoint differs from its committed integrity record.')
    if expected_sha256 is not None and digest != expected_sha256:
        raise ValueError('Checkpoint SHA256 differs from the frozen identity.')
    # Project-owned torch checkpoints contain optimizer/RNG objects. Do not pass unknown pickle files.
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(payload, dict) or 'model' not in payload:
        raise ValueError('Expected an AHR checkpoint containing model weights.')
    extra = payload.get('extra', {})
    training_stage = extra.get('stage', extra.get('training_stage', payload.get('training_stage')))
    if training_stage not in {'supervised', 'joint'} or int(payload.get('step', 0)) < 1:
        raise ValueError('A trained supervised/joint checkpoint is required; recovery fixtures and step-zero weights are not trained agents.')
    model_cfg = extra.get('model_config', payload.get('config'))
    if not isinstance(model_cfg, dict) or 'model' not in model_cfg:
        raise ValueError('Checkpoint lacks the full model configuration.')
    identity = {'path': str(path), 'sha256': digest, 'step': int(payload.get('step', 0)), 'training_stage': training_stage}
    return payload, copy.deepcopy(model_cfg), identity


def load_training_rows(cfg, stage, root=ROOT):
    registry = SplitRegistry.from_config(cfg)
    if not cfg['data'].get('train_games'):
        raise ValueError('Set explicit data.train_games and disjoint dev/heldout game families in notebook 03.')
    candidates = [cfg['data']['replay_path']]
    if stage == 'supervised':
        candidates.append(cfg['data']['expert_path'])
    paths, rows, identities, seen = [], [], {}, set()
    for value in dict.fromkeys(candidates):
        path = Path(value)
        if not path.is_absolute():
            path = Path(root) / path
        if not path.is_file():
            continue
        records = load_transitions(path, split='train')
        identities[str(path.relative_to(root)) if path.is_relative_to(root) else str(path)] = sha256_file(path)
        paths.append(path)
        for row in records:
            registry.assert_game(row['game_id'], 'train')
            identity = (row['episode_id'], int(row['step']), row['game_id'])
            if identity in seen:
                raise ValueError(f'Duplicate transition across training inputs: {identity}')
            seen.add(identity)
            rows.append(row)
    if not rows:
        raise FileNotFoundError('No validated train transitions. Collect replay and/or explicit demonstrations with notebook 03.')
    if stage == 'supervised':
        supervised_count = sum(bool(r.get('teacher_action')) and r.get('source') in {'human_demonstration', 'expert_demonstration'} and bool(r.get('provenance', {}).get('teacher_id')) for r in rows)
        if supervised_count == 0:
            raise ValueError('Supervised stage needs explicit demonstrations with teacher_action and provenance.teacher_id; random actions are not teachers.')
    split_path = Path(cfg['data']['split_manifest'])
    if not split_path.is_absolute():
        split_path = Path(root) / split_path
    if not split_path.is_file():
        raise FileNotFoundError('Run notebook 03 to create the split manifest before training.')
    manifest_registry = SplitRegistry.from_manifest(split_path)
    if manifest_registry.to_dict() != registry.to_dict():
        raise ValueError('Saved split manifest and current configuration disagree.')
    identities['split_manifest'] = sha256_file(split_path)
    return rows, {'files': identities, 'rows': len(rows), 'order_hash': canonical_hash([(r['episode_id'], r['step'], r['game_id']) for r in rows])}


def training_precision(cfg, device):
    precision = cfg['train'].get('precision', 'bf16')
    if precision not in {'bf16', 'fp16', 'fp32'}:
        raise ValueError('precision must be bf16, fp16 or fp32')
    if device.type != 'cuda' or precision == 'fp32':
        return contextlib.nullcontext()
    return torch.autocast('cuda', dtype=torch.bfloat16 if precision == 'bf16' else torch.float16)


def legal_policy(logits, masks):
    if not masks.any(dim=-1).all():
        raise ValueError('A nonterminal training observation has no available actions.')
    return logits.float().masked_fill(~masks.bool(), -1e9)


def action_log_loss(logits, click_logits, actions, xy_targets, masks):
    if not masks.gather(1, actions[:, None]).all():
        raise ValueError('A training label violates the observed action mask.')
    result = F.cross_entropy(legal_policy(logits, masks), actions, reduction='none')
    clicks = actions == 6
    if clicks.any():
        result = result.clone()
        result[clicks] += F.cross_entropy(click_logits[clicks, 0].float(), xy_targets[clicks, 0], reduction='none')
        result[clicks] += F.cross_entropy(click_logits[clicks, 1].float(), xy_targets[clicks, 1], reduction='none')
    return result


def transition_objective(model, batch, cfg, stage):
    out = model(batch['grids'], actions=batch['actions'], xy=batch['xy'])
    with torch.no_grad():
        _, next_latent = model.encoder(batch['next_grids'])
        next_latent = next_latent.float()
        next_value = model(batch['next_grids'])['value'].float().reshape(-1)
    latent = out['latent'].float()
    goal = out['goal'].float()
    predicted = out['next_latent'].float()
    variance = out['uncertainty'].float().reshape(-1).clamp(1e-4, 1e4)
    error = (predicted - next_latent).square().mean(-1)
    dynamics = 0.5 * (error / variance + variance.log()).mean()
    reward_loss = F.smooth_l1_loss(out['reward'].float().reshape(-1), batch['rewards'].float())
    done_loss = F.binary_cross_entropy_with_logits(out['done_logit'].float().reshape(-1), batch['dones'].float())
    # In-batch negatives plus a perturbation avoid using the identical positive when B=1.
    negative = next_latent.roll(1, dims=0) + 0.05 * torch.randn_like(next_latent)
    worker_positive = model.worker.energy(latent, next_latent, goal).reshape(-1)
    worker_negative = model.worker.energy(latent, negative, goal).reshape(-1)
    worker_rank = F.softplus(1.0 + worker_positive - worker_negative).mean()
    worker_error = (out['worker_proposal'].float() - next_latent).square().mean(-1)
    scout_enabled = cfg['reasoning'].get('scout_enabled', True)
    if scout_enabled:
        scout_positive = model.scout.energy(latent, next_latent, goal).reshape(-1)
        scout_negative = model.scout.energy(latent, negative, goal).reshape(-1)
        scout_rank = F.softplus(1.0 + scout_positive - scout_negative).mean()
        scout_error = (out['scout_proposal'].float() - next_latent).square().mean(-1)
    else:
        scout_rank = latent.new_zeros(())
        scout_error = torch.zeros_like(worker_error)
    proposal_loss = worker_error.mean() + scout_error.mean()
    if cfg['reasoning'].get('queen_mode', 'learned') == 'learned':
        mixture = out['worker_weight'].float().reshape(-1) * worker_error
        if scout_enabled:
            mixture = mixture + out['scout_weight'].float().reshape(-1) * scout_error
        schedule_loss = mixture.mean()
        halt_loss = F.binary_cross_entropy_with_logits(out['halt_logit'].float().reshape(-1), batch['dones'].float())
    else:
        schedule_loss = halt_loss = latent.new_zeros(())
    geometry_loss = model.geometry.regularizer(latent)
    geometry_loss = geometry_loss + F.mse_loss(model.geometry.embed(predicted), model.geometry.embed(next_latent).detach())
    # This auxiliary anti-collapse term is meaningful only when the batch has variation.
    variance_loss = F.relu(0.1 - latent.std(dim=0, unbiased=False)).mean() if latent.shape[0] > 1 else latent.new_zeros(())
    gamma = float(cfg['train'].get('gamma', 0.99))
    target_value = batch['rewards'].float() + gamma * (1.0 - batch['dones'].float()) * next_value
    values = out['value'].float().reshape(-1)
    value_loss = F.smooth_l1_loss(values, target_value)
    teacher_mask = batch['teacher_mask'].bool()
    teacher_loss = latent.new_zeros(())
    if teacher_mask.any():
        teacher_loss = action_log_loss(out['action_logits'][teacher_mask], out['click_logits'][teacher_mask], batch['teacher_actions'][teacher_mask], batch['teacher_click_targets'][teacher_mask], batch['masks'][teacher_mask]).mean()
    logits = legal_policy(out['action_logits'], batch['masks'])
    probabilities = logits.softmax(-1)
    entropy = -(probabilities * logits.log_softmax(-1)).sum(-1).mean()
    actor_loss = latent.new_zeros(())
    if stage == 'joint':
        # Advantage-weighted regression on logged actions; this is OFFLINE, not on-policy PPO.
        advantage = (target_value - values).detach()
        temperature = max(float(cfg['train'].get('advantage_temperature', 1.0)), 1e-3)
        weights = torch.exp((advantage / temperature).clamp(-5, math.log(20.0)))
        weights = weights / weights.mean().clamp_min(1e-6)
        action_xy = ((batch['xy'].float() + 1.0) * 31.5).round().clamp(0, 63).long()
        actor_loss = (weights * action_log_loss(out['action_logits'], out['click_logits'], batch['actions'], action_xy, batch['masks'])).mean()
    total = (teacher_loss + actor_loss + dynamics + reward_loss + done_loss + 0.5 * value_loss
             + 0.1 * (worker_rank + scout_rank + proposal_loss + schedule_loss + halt_loss)
             + 0.01 * (geometry_loss + variance_loss) - 0.001 * entropy)
    components = {'loss': total, 'teacher': teacher_loss, 'actor': actor_loss, 'dynamics': dynamics,
                  'reward': reward_loss, 'done': done_loss, 'value': value_loss, 'worker_energy': worker_rank,
                  'scout_energy': scout_rank, 'proposal': proposal_loss, 'schedule': schedule_loss,
                  'geometry': geometry_loss, 'entropy': entropy, 'teacher_fraction': teacher_mask.float().mean()}
    return total, components


## Bounded optimizer and exact continuation
The best checkpoint uses the lowest smoothed **training objective among scheduled checkpoint boundaries**, not a held-out score. A benchmark or development checkpoint selection policy is a later refinement. A failure during a partially completed update preserves the last verified boundary checkpoint rather than claiming that an uncertain optimizer state is resumable.

In [ ]:
def train_stage(cfg, run_id, stage='supervised', root=ROOT):
    if stage not in {'supervised', 'joint'}:
        raise ValueError('Unknown training stage.')
    cfg = copy.deepcopy(cfg)
    cfg['train']['stage'] = stage
    require_recovery_gate(root)
    require_full_model_gate(cfg, root)
    rows, data_identity = load_training_rows(cfg, stage, root)
    device = torch.device(cfg['train']['device'])
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Selected CUDA device is unavailable. Run notebook 00 hardware checks.')
    max_steps = int(cfg['train']['max_steps'])
    batch_size = int(cfg['train']['batch_size'])
    wall_limit = float(cfg['train']['max_wall_seconds'])
    if min(max_steps, batch_size) < 1 or wall_limit <= 0:
        raise ValueError('Positive step, batch and wall-clock limits are required.')
    warmstart = cfg['train'].get('warmstart_checkpoint')
    if stage == 'joint' and not warmstart:
        raise ValueError('Set train.warmstart_checkpoint to an explicit supervised checkpoint before joint training.')
    warm_payload, warm_identity = None, None
    if warmstart:
        warm_payload, warm_cfg, warm_identity = read_ahr_checkpoint(warmstart, root, cfg['train'].get('warmstart_sha256'))
        if warm_cfg['model'] != cfg['model']:
            raise ValueError('Warm-start model dimensions differ; architecture migration must be explicit.')
        if int(warm_payload.get('step', 0)) < 1:
            raise ValueError('Warm start must contain at least one completed training update.')
    training_families = {game.split('-', 1)[0] for game in cfg['data']['train_games']}
    if warm_payload is not None:
        warm_extra = warm_payload.get('extra', {})
        warm_training_cfg = warm_extra.get('trained_config', warm_cfg)
        training_families.update(game.split('-', 1)[0] for game in warm_training_cfg.get('data', {}).get('train_games', []))
        training_families.update(warm_extra.get('training_game_families', []))
    training_families = sorted(training_families)
    seed_everything(int(cfg['train']['seed']))
    manager_cfg = {'model_config': cfg, 'train': cfg['train'], 'stage': stage, 'data_identity': data_identity,
                   'warmstart_identity': warm_identity, 'runtime': runtime_fingerprint()}
    with RunLock(root, run_id):
        seed_everything(int(cfg['train']['seed']))
        model = AHRModel(cfg).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg['train']['lr']), weight_decay=float(cfg['train']['weight_decay']))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: max(0.05, 0.5 * (1 + math.cos(math.pi * min(s, max_steps) / max_steps))))
        scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['train'].get('precision') == 'fp16')
        sampler = StatefulSampler(len(rows), int(cfg['train']['seed']))
        manager = CheckpointManager(root, run_id, manager_cfg, source_hashes=source_manifest(root))
        payload = manager.load_latest(model, optimizer, scheduler, scaler, map_location=device)
        step, best, moving, latest = 0, float('inf'), None, {}
        if payload is not None:
            step = int(payload['step'])
            sampler.load_state_dict(payload['extra']['sampler'])
            best = float(payload['extra']['best_training_loss'])
            moving = payload['extra'].get('moving_training_loss')
            latest = payload.get('metrics', {})
            print(f'Resumed {run_id}: completed optimizer step {step}, metrics={latest}')
        elif warm_payload is not None:
            model.load_state_dict(warm_payload['model'], strict=True)
            print(f'Warm start from {warm_identity["path"]}, step {warm_identity["step"]}, SHA256 {warm_identity["sha256"]}')
        elif stage != 'supervised':
            raise RuntimeError('Joint training cannot silently start from random weights.')
        metrics_path = Path(root) / 'metrics' / f'{run_id}.jsonl'
        if payload is not None and metrics_path.exists():
            # Metrics beyond the resumed optimizer boundary are abandoned, not double-counted.
            retained = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
            retained = [row for row in retained if int(row.get('step', -1)) <= step]
            temporary_metrics = metrics_path.with_suffix('.jsonl.tmp')
            with temporary_metrics.open('w') as stream:
                for row in retained:
                    stream.write(json.dumps(row, allow_nan=False) + '\n')
                stream.flush()
                os.fsync(stream.fileno())
            os.replace(temporary_metrics, metrics_path)
        started = time.monotonic()
        old_epoch = (step * batch_size) // len(rows)
        def save_boundary(is_best=None, status='running'):
            nonlocal best
            if is_best is None:
                is_best = moving is not None and moving < best
            if is_best:
                best = moving
            extra = {'sampler': sampler.state_dict(), 'data_identity': data_identity, 'model_config': cfg,
                     'best_training_loss': best, 'moving_training_loss': moving, 'stage': stage,
                     'training_game_families': training_families,
                     'warmstart_identity': warm_identity, 'status': status,
                     'online_environment': None, 'replay_mode': 'immutable_observed_transition_files'}
            return manager.save(model, optimizer, scheduler, scaler, step=step, metrics=latest, extra=extra, is_best=is_best)
        safe_boundary = True
        save_boundary(status='ready')
        model.train()
        interrupted = False
        try:
            while step < max_steps and time.monotonic() - started < wall_limit:
                safe_boundary = False
                indices = sampler.next_indices(batch_size)
                batch = transition_batch([rows[int(i)] for i in indices], device)
                optimizer.zero_grad(set_to_none=True)
                with training_precision(cfg, device):
                    loss, values = transition_objective(model, batch, cfg, stage)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite objective; preserving the last verified checkpoint.')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['train']['grad_clip']), error_if_nonfinite=True)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                if device.type == 'cuda':
                    torch.cuda.synchronize(device)
                step += 1
                latest = {k: float(v.detach().float().cpu()) for k, v in values.items()}
                latest.update({'step': step, 'grad_norm': float(norm), 'lr': optimizer.param_groups[0]['lr'],
                               'elapsed_seconds': time.monotonic() - started, 'stage': stage})
                moving = latest['loss'] if moving is None else 0.95 * moving + 0.05 * latest['loss']
                safe_boundary = True
                append_metrics(metrics_path, latest)
                epoch = (step * batch_size) // len(rows)
                # Explicit cadence is capped even if a future manager default becomes looser.
                if step == 1 or manager.due(step) or step % 200 == 0 or epoch != old_epoch:
                    save_boundary()
                old_epoch = epoch
                if step == 1 or step % 10 == 0:
                    print({k: latest[k] for k in ('step', 'loss', 'teacher', 'actor', 'dynamics')})
        except KeyboardInterrupt:
            interrupted = True
            if safe_boundary:
                save_boundary(status='interrupted_at_boundary')
            else:
                print('Interrupted inside an update; resume the previous verified checkpoint. The partial update is discarded.')
        except BaseException:
            if safe_boundary:
                save_boundary(status='failed_at_boundary')
            raise
        finally:
            if safe_boundary:
                save_boundary(status='interrupted' if interrupted else ('target_reached' if step >= max_steps else 'wall_limit'))
        report = {'run_id': run_id, 'stage': stage, 'completed_step_in_memory': step,
                  'status': 'interrupted' if interrupted else ('target_reached' if step >= max_steps else 'wall_limit'),
                  'checkpoint_directory': str(Path(root) / 'checkpoints' / run_id), 'metrics_path': str(metrics_path),
                  'data_identity': data_identity, 'model_config': cfg, 'source_hashes': source_manifest(root),
                  'best_selection': 'minimum smoothed training objective among scheduled checkpoint boundaries; not benchmark performance',
                  'safe_boundary': safe_boundary, 'metrics': latest, 'runtime': runtime_fingerprint()}
        atomic_json(Path(root) / 'reports' / f'{run_id}.json', report)
        return report


def train_supervised(cfg, run_id, root=ROOT):
    return train_stage(cfg, run_id, stage='supervised', root=root)


def training_readiness(cfg, stage, root=ROOT):
    problems = []
    for checker in (lambda: require_recovery_gate(root), lambda: require_full_model_gate(cfg, root), lambda: load_training_rows(cfg, stage, root)):
        try:
            checker()
        except (FileNotFoundError, ValueError, RuntimeError) as exc:
            problems.append(str(exc))
    if stage == 'joint' and not cfg['train'].get('warmstart_checkpoint'):
        problems.append('Set train.warmstart_checkpoint to the explicit supervised checkpoint chosen for joint training.')
    return problems


## Configure and run this stage
Edit the shared configuration in notebook 00 before validation in notebook 09. Choose a new run ID for changed hyperparameters or data; unchanged IDs resume. Running this cell is a real training request once prerequisites are satisfied.

In [ ]:
cfg = load_config()
cfg['train']['stage'] = 'supervised'
RUN_ID = 'xl-supervised-seed42-v1'
problems = training_readiness(cfg, 'supervised')
if problems:
    print('SUPERVISED TRAINING NOT STARTED. Resolve these prerequisites:')
    for problem in problems:
        print(' •', problem)
else:
    supervised_report = train_supervised(cfg, RUN_ID)
    print(supervised_report['status'], supervised_report['checkpoint_directory'])


## Inspect measured learning curves after a run
Loss curves describe optimization; they do not establish reasoning ability or game generalization.

In [ ]:
curve_path = ROOT / 'metrics' / f'{RUN_ID}.jsonl'
if curve_path.exists():
    import matplotlib.pyplot as plt
    history = [json.loads(line) for line in curve_path.read_text().splitlines() if line.strip()]
    if history:
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.plot([r['step'] for r in history], [r['loss'] for r in history], label='Observed training objective')
        ax.set(xlabel='Completed optimizer step', ylabel='Objective', title=RUN_ID)
        ax.grid(alpha=0.2)
        ax.legend()
        plt.show()
else:
    print('No learning curve yet; this notebook remains unexecuted until Run All is requested.')
